In [1]:
import requests
import pandas as pd
import time

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAQ_API_KEY")

In [3]:
HEADERS = {"X-API-Key": api_key}
BASE_URL = "https://api.openaq.org/v3"
COUNTRY_ID = 145  

In [4]:
resp = requests.get(
    f"{BASE_URL}/countries",
    headers=HEADERS,
    params={"limit": 1000}
)
countries = resp.json()["results"]
nepal = [c for c in countries if c["code"] == "NP"]
print(nepal)

[{'id': 145, 'code': 'NP', 'name': 'Nepal', 'datetimeFirst': '2017-03-03T00:00:00Z', 'datetimeLast': '2026-07-23T04:00:00Z', 'parameters': [{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, {'id': 98, 'name': 'relativehumidity', 'units': '%', 'displayName': None}, {'id': 100, 'name': 'temperature', 'units': 'c', 'displayName': None}, {'id': 125, 'name': 'um003', 'units': 'particles/cm³', 'displayName': None}]}]


In [5]:
country_id = 145

def get_nepal_locations(country_id):
    locations = []
    page = 1
    while True:
        resp = requests.get(
            f"{BASE_URL}/locations",
            headers=HEADERS,
            params={"countries_id": country_id, "limit": 1000, "page": page}
        )
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])
        if not results:
            break
        locations.extend(results)
        if len(results) < 1000:
            break
        page += 1
        time.sleep(0.5)
    return locations

nepal_locations = get_nepal_locations(country_id)
print(f"Found {len(nepal_locations)} stations in Nepal")

Found 86 stations in Nepal


In [13]:
def get_all_sensors(locations):
    """Get sensor_id + parameter name for every station"""
    sensor_list = []
    for loc in locations:
        for s in loc.get("sensors", []):
            sensor_list.append({
                "sensor_id": s["id"],
                "parameter": s["parameter"]["name"],
                "location_id": loc["id"],
                "location_name": loc["name"]
            })
    return sensor_list

all_sensors = get_all_sensors(nepal_locations)
print(f"Total sensors across Nepal: {len(all_sensors)}")

Total sensors across Nepal: 401


In [10]:
resp = requests.get(
    f"{BASE_URL}/sensors/7710/days/monthly",
    headers=HEADERS,
    params={"datetime_from": "2017-01-01", "datetime_to": "2026-07-14", "limit": 1000}
)
print(resp.status_code)
print(resp.json())

200
{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 1000, 'found': 108}, 'results': [{'value': 62.5, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1 month', 'interval': '1 month', 'datetimeFrom': {'utc': '2017-02-28T18:15:00Z', 'local': '2017-03-01T00:00:00+05:45'}, 'datetimeTo': {'utc': '2017-03-31T18:15:00Z', 'local': '2017-04-01T00:00:00+05:45'}}, 'coordinates': None, 'summary': {'min': 23.84166717529297, 'q02': 24.892833669026693, 'q25': 49.01145839691162, 'median': 58.9375, 'q75': 65.28370370370371, 'q98': 132.51036645507813, 'max': 134.51666666666665, 'avg': 62.51678046056721, 'sd': 28.345939314146}, 'coverage': {'expectedCount': 31, 'expectedInterval': '744:00:00', 'observedCount': 22, 'observedInterval': '528:00:00', 'percentComplete': 71.0, 'percentCoverage': 71.0, 'datetimeFrom': {'utc': '2017-03-02T18:15:00Z', 'local': '2017-03-03T00:00:00+05:45'}, 'datetimeTo': {'utc':

In [15]:
import requests
import pandas as pd
import time

def get_monthly_data(sensor_id, retries=2):
    all_rows = []
    page = 1
    while True:
        resp = requests.get(
            f"{BASE_URL}/sensors/{sensor_id}/days/monthly",
            headers=HEADERS,
            params={"datetime_from": "2017-01-01", "datetime_to": "2026-07-14", "limit": 1000, "page": page}
        )
        if resp.status_code == 429:
            time.sleep(10)
            continue
        if resp.status_code != 200:
            return all_rows  # skip broken sensors, don't crash the whole loop
        data = resp.json()
        results = data.get("results", [])
        if not results:
            break
        all_rows.extend(results)
        if len(results) < 1000:
            break
        page += 1
    return all_rows

# Build the sensor list first (you already have all_sensors from before)
all_records = []
for i, s in enumerate(all_sensors):
    sensor_id = s["sensor_id"]
    rows = get_monthly_data(sensor_id)
    for r in rows:
        r["sensor_id"] = sensor_id
        r["parameter_name"] = s["parameter"]
        r["location_id"] = s["location_id"]
        r["location_name"] = s["location_name"]
    all_records.extend(rows)
    if i % 20 == 0:
        print(f"{i}/{len(all_sensors)} sensors done, total rows so far: {len(all_records)}")
    time.sleep(0.3)

print(f"\nTOTAL ROWS: {len(all_records)}")

0/401 sensors done, total rows so far: 85
20/401 sensors done, total rows so far: 661
40/401 sensors done, total rows so far: 831
60/401 sensors done, total rows so far: 1006
80/401 sensors done, total rows so far: 1166
100/401 sensors done, total rows so far: 1346
120/401 sensors done, total rows so far: 1521
140/401 sensors done, total rows so far: 1701
160/401 sensors done, total rows so far: 1901
180/401 sensors done, total rows so far: 2096
200/401 sensors done, total rows so far: 2251
220/401 sensors done, total rows so far: 2451
240/401 sensors done, total rows so far: 2616
260/401 sensors done, total rows so far: 2786
280/401 sensors done, total rows so far: 2941
300/401 sensors done, total rows so far: 3116
320/401 sensors done, total rows so far: 3281
340/401 sensors done, total rows so far: 3371
360/401 sensors done, total rows so far: 3426
380/401 sensors done, total rows so far: 3486
400/401 sensors done, total rows so far: 3521

TOTAL ROWS: 3521


In [16]:
df_check = pd.DataFrame(all_records)
df_check.groupby('location_name').size().sort_values(ascending=False).head(20)

location_name
Phora Durbar Kathman                                              194
Embassy Kathmandu                                                 193
Dabali, Handigaun                                                 134
Dhathutole, Handigaun                                              90
Bharatpur Ward no 27 office Meghauli                               55
Chovar (SC - 07) - GD Labs                                         55
Hetauda Udhyog Sang Office (CEN-SR-18)                             55
Sunakothi (SC - 06) - GD Labs                                      55
CEN-SR-12: Lamahi Municipality Office, Dang                        50
Baluwatar (SC-02) - GD Labs                                        50
CEN-SR-08/ Birendranagar Municipality                              50
CEN-SR-10: Siddharthanagar Municipality Ward no. 4                 50
CEN-SR-07: Siddharthanagar Municipality City Office Ward No. 5     50
CEN-SR-22:Birendranagar Ward 12 Sahakari Chowk                     50
CEN_SR

In [17]:
df_check

,value,flagInfo,parameter,period,coordinates,summary,coverage,sensor_id,parameter_name,location_id,location_name
0,-0.22100,{'hasFlags': False},"{'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': -0.9989999999999999, 'q02': -0.9989999...","{'expectedCount': 31, 'expectedInterval': '744...",7713,o3,3459,Embassy Kathmandu
1,-0.15900,{'hasFlags': True},"{'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': -0.9989999999999998, 'q02': -0.9989999...","{'expectedCount': 30, 'expectedInterval': '720...",7713,o3,3459,Embassy Kathmandu
2,-0.14200,{'hasFlags': True},"{'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': -0.9989999999999998, 'q02': -0.9989999...","{'expectedCount': 31, 'expectedInterval': '744...",7713,o3,3459,Embassy Kathmandu
3,-0.02790,{'hasFlags': True},"{'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': -0.9989999999999998, 'q02': -0.7479808...","{'expectedCount': 30, 'expectedInterval': '720...",7713,o3,3459,Embassy Kathmandu
4,-0.00133,{'hasFlags': True},"{'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': -0.07519230769230771, 'q02': -0.044715...","{'expectedCount': 31, 'expectedInterval': '744...",7713,o3,3459,Embassy Kathmandu
...,...,...,...,...,...,...,...,...,...,...,...
3516,12.70000,{'hasFlags': False},"{'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'd...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': 6.154545454545455, 'q02': 6.4586363636...","{'expectedCount': 31, 'expectedInterval': '744...",16888521,pm1,6439605,"FHI-OU-05: Nayagaun Secondary School, BSMC"
3517,25.50000,{'hasFlags': False},"{'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'd...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': 16.42727272727273, 'q02': 16.863909090...","{'expectedCount': 31, 'expectedInterval': '744...",16888522,pm25,6439605,"FHI-OU-05: Nayagaun Secondary School, BSMC"
3518,62.60000,{'hasFlags': False},"{'id': 98, 'name': 'relativehumidity', 'units'...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': 61.0, 'q02': 61.057272727272725, 'q25'...","{'expectedCount': 31, 'expectedInterval': '744...",16888523,relativehumidity,6439605,"FHI-OU-05: Nayagaun Secondary School, BSMC"
3519,30.30000,{'hasFlags': False},"{'id': 100, 'name': 'temperature', 'units': 'c...","{'label': '1 month', 'interval': '1 month', 'd...",None,"{'min': 29.650000000000002, 'q02': 29.67645454...","{'expectedCount': 31, 'expectedInterval': '744...",16888524,temperature,6439605,"FHI-OU-05: Nayagaun Secondary School, BSMC"


In [18]:
df_air = pd.DataFrame(all_records)
print(df_air.shape)

(3521, 11)


In [ ]:
df_air.to_csv("dv_group_nepal_air_quality_monthly.csv", index=False)


# Weather

In [27]:
nepal_locations = get_nepal_locations(country_id)
print(f"Found {len(nepal_locations)} stations in Nepal")

Found 81 stations in Nepal


In [28]:
import requests
import pandas as pd
import time

def get_nasa_power_data(lat, lon, start="20170101", end="20260714"):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M,PRECTOTCORR,RH2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON"
    }
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        print(f"  failed for {lat},{lon}: {resp.status_code}")
        return None
    return resp.json()

weather_records = []
seen_coords = set()

for loc in nepal_locations:
    lat = loc["coordinates"]["latitude"] if loc.get("coordinates") else None
    lon = loc["coordinates"]["longitude"] if loc.get("coordinates") else None
    if lat is None or lon is None:
        continue
    coord_key = (round(lat, 2), round(lon, 2))
    if coord_key in seen_coords:
        continue
    seen_coords.add(coord_key)

    print(f"Pulling weather for {loc['name']} ({lat}, {lon})...")
    data = get_nasa_power_data(lat, lon)
    if data is None:
        continue

    params_data = data.get("properties", {}).get("parameter", {})
    dates = params_data.get("T2M", {}).keys()
    for date in dates:
        weather_records.append({
            "location_name": loc["name"],
            "latitude": lat,
            "longitude": lon,
            "date": date,
            "temperature_c": params_data.get("T2M", {}).get(date),
            "precipitation_mm": params_data.get("PRECTOTCORR", {}).get(date),
            "humidity_pct": params_data.get("RH2M", {}).get(date),
        })
    time.sleep(1)

Pulling weather for Embassy Kathmandu (27.738703, 85.336206)...
Pulling weather for Phora Durbar Kathman (27.712464, 85.315703)...
Pulling weather for Dhathutole, Handigaun (27.727502, 85.330135)...
Pulling weather for Dabali, Handigaun (27.68189114, 85.28707804)...
Pulling weather for Gaushala Chowk (SC-01) - GD Labs (27.707763, 85.343189)...
Pulling weather for Baluwatar (SC-02) - GD Labs (27.7249438, 85.331062)...
Pulling weather for Lagankhel (SC - 05) - GD Labs (27.666459, 85.323093)...
Pulling weather for Lamtangil (SC-04)- GD Labs (27.731368, 85.336783)...
Pulling weather for Pulchowk (SC-44) - GD Labs (27.6773887, 85.3184176)...
Pulling weather for Nakhipot (SC-08) - GD Labs (27.6510811, 85.3178568)...
Pulling weather for Sunakothi (SC - 06) - GD Labs (27.630412, 85.320402)...
Pulling weather for Chovar (SC - 07) - GD Labs (27.6662854, 85.2937189)...
Pulling weather for Taudaha (SC - 09) - GD Labs (27.651121, 85.283954)...
Pulling weather for CEN-SR-25: Patako Chowk, Patan Durb

In [29]:
df_weather = pd.DataFrame(weather_records)
print(df_weather.shape)

(240258, 7)


In [31]:
df_weather.to_csv("dv_group_weather_daily.csv", index=False)

# Tourism 

In [32]:
!pip install pypdf


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pypdf

reader = pypdf.PdfReader('Nepal tourism statistics 2025_ibddtm6.pdf')

# Table 2.2 spans pages 29 and 30 (0-indexed)
page1_text = reader.pages[29].extract_text()
page2_text = reader.pages[30].extract_text()

print(page1_text)
print(page2_text)

12 | NEPAL TOURISM STATISTICS, 2025
TABLE 2.2: TOURIST ARRIVAL BY MONTH, 1995-2025
Y ear Jan. Feb. Mar. Apr. May Jun. Jul. Aug. Sep. Oct. Nov. Dec. Total
1995 22,207 28,240 34,219 33,994 27,843 25,650 23,980 27,686 30,569 46,845 35,782 26,380 363,395
1996 27,886 29,676 39,336 36,331 29,728 26,749 22,684 29,080 32,181 47,314 37,650 34,998 393,613
1997 25,585 32,861 43,177 35,229 33,456 26,367 26,091 35,549 31,981 56,272 40,173 35,116 421,857
1998 28,822 37,956 41,338 41,087 35,814 29,181 27,895 36,174 39,664 62,487 47,403 35,863 463,684
1999 29,752 38,134 46,218 40,774 42,712 31,049 27,193 38,449 44,117 66,543 48,865 37,698 491,504
2000 25,307 38,959 44,944 43,635 28,363 26,933 24,480 34,670 43,523 59,195 52,993 40,644 463,646
2001 30,454 38,680 46,709 39,083 28,345 13,030 18,329 25,322 31,170 41,245 30,282 18,588 361,237
2002 17,176 20,668 28,815 21,253 19,887 17,218 16,621 21,093 23,752 35,272 28,723 24,990 275,468
2003 21,215 24,349 27,737 25,851 22,704 20,351 22,661 27,568 28,724 45

In [21]:
raw_2021_2025 = """2021 8,874 9,266 15,254 22,732 1,531 1,187 3,093 6,093 9,907 23,338 26,135 23,552 150,962
2022 16975 19856 42152 61589 54093 46957 44462 41304 58314 88582 72653 67932 614869
2023 55074 73255 99426 98773 77703 72250 57726 67153 91012 117306 108630 96574 1014882
2024 79101 97423 128167 111382 90205 76733 64598 72717 96302 124391 114496 92033 1147548
2025 79990 99585 121632 116487 86216 76425 70193 88681 79981 128443 116552 98180 1162365"""

In [22]:
import pandas as pd

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
records = []
for line in raw_2021_2025.strip().split('\n'):
    parts = line.split()
    year = int(parts[0])
    nums = [int(p.replace(',','')) for p in parts[1:13]]
    for m, val in zip(months, nums):
        records.append({'year': year, 'month': m, 'tourist_arrivals': val})

df_tourism = pd.DataFrame(records)

In [23]:
df_tourism

,year,month,tourist_arrivals
0,2021,Jan,8874
1,2021,Feb,9266
2,2021,Mar,15254
3,2021,Apr,22732
4,2021,May,1531
5,2021,Jun,1187
6,2021,Jul,3093
7,2021,Aug,6093
8,2021,Sep,9907
9,2021,Oct,23338


In [24]:
print(df_tourism.shape)  # should be (60, 3)

(60, 3)


In [25]:
df_tourism.to_csv('nepal_tourism_monthly_2021_2025.csv', index=False)

In [32]:
import pandas as pd
df_air = pd.read_csv('dv_group_nepal_air_quality_monthly.csv')
print(df_air.columns.tolist())
print(df_air.head(2))

['value', 'flagInfo', 'parameter', 'period', 'coordinates', 'summary', 'coverage', 'sensor_id', 'parameter_name', 'location_id', 'location_name']
   value             flagInfo  \
0 -0.221  {'hasFlags': False}   
1 -0.159   {'hasFlags': True}   

                                           parameter  \
0  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   
1  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   

                                              period  coordinates  \
0  {'label': '1 month', 'interval': '1 month', 'd...          NaN   
1  {'label': '1 month', 'interval': '1 month', 'd...          NaN   

                                             summary  \
0  {'min': -0.9989999999999999, 'q02': -0.9989999...   
1  {'min': -0.9989999999999998, 'q02': -0.9989999...   

                                            coverage  sensor_id  \
0  {'expectedCount': 31, 'expectedInterval': '744...       7713   
1  {'expectedCount': 30, 'expectedInterval': '720...       7713   

  para

In [33]:
df_weather = pd.read_csv('nepal_weather_daily.csv')
df_weather['year'] = df_weather['date'].astype(str).str[:4].astype(int)
df_weather_filtered = df_weather[(df_weather['year'] >= 2021) & (df_weather['year'] <= 2025)]
print(df_weather_filtered.shape)
df_weather_filtered.to_csv('nepal_weather_daily_2021_2025.csv', index=False)

(125994, 8)


In [34]:
import ast

df_air = pd.read_csv('dv_group_nepal_air_quality_monthly.csv')

# The period column is a stringified dict — parse it safely
def get_year(period_str):
    try:
        period_dict = ast.literal_eval(period_str)
        date_str = period_dict['datetimeFrom']['local']  # e.g. '2021-01-01T00:00:00+05:45'
        return int(date_str[:4])
    except:
        return None

df_air['year'] = df_air['period'].apply(get_year)
df_air_filtered = df_air[(df_air['year'] >= 2021) & (df_air['year'] <= 2025)]
print(df_air_filtered.shape)
df_air_filtered.to_csv('nepal_air_quality_monthly_2021_2025.csv', index=False)

(1191, 12)


In [35]:
#Aggregation

In [36]:
df_weather = pd.read_csv('nepal_weather_daily_2021_2025.csv')
df_weather['date'] = pd.to_datetime(df_weather['date'].astype(str), format='%Y%m%d')
df_weather['year'] = df_weather['date'].dt.year
df_weather['month'] = df_weather['date'].dt.strftime('%b')  # Jan, Feb, etc.

df_weather_monthly = df_weather.groupby(['location_name', 'year', 'month']).agg(
    temperature_c=('temperature_c', 'mean'),
    precipitation_mm=('precipitation_mm', 'sum'),
    humidity_pct=('humidity_pct', 'mean')
).reset_index()

print(df_weather_monthly.shape)

(4140, 6)


In [37]:
import ast

def get_month_year(period_str):
    d = ast.literal_eval(period_str)
    date_str = d['datetimeFrom']['local']
    dt = pd.to_datetime(date_str)
    return dt.year, dt.strftime('%b')

df_air = pd.read_csv('nepal_air_quality_monthly_2021_2025.csv')
df_air[['year','month']] = df_air['period'].apply(lambda x: pd.Series(get_month_year(x)))

# Keep only what you need
df_air_clean = df_air[['location_name','year','month','parameter_name','value']]

# Pivot so pm25/o3 become their own columns
df_air_pivot = df_air_clean.pivot_table(
    index=['location_name','year','month'],
    columns='parameter_name',
    values='value'
).reset_index()

print(df_air_pivot.shape)
print(df_air_pivot.head())

(306, 10)
parameter_name                location_name  year month  o3   pm1  pm10  \
0                   Balaju (SC-26)- GD Labs  2025   Dec NaN  62.6   NaN   
1                   Balaju (SC-26)- GD Labs  2025   Oct NaN  26.5   NaN   
2                 Balkumari(SC-28)- GD Labs  2025   Dec NaN  52.7   NaN   
3                 Balkumari(SC-28)- GD Labs  2025   Nov NaN  32.6   NaN   
4               Baluwatar (SC-02) - GD Labs  2025   Dec NaN  54.0   NaN   

parameter_name   pm25  relativehumidity  temperature   um003  
0               103.0              44.2         19.1  4580.0  
1                41.7              42.4         26.6  1710.0  
2                85.9              39.6         20.9  3850.0  
3                53.1              42.1         22.1  2320.0  
4                83.9              42.7         19.0  3760.0  


In [38]:
df_merged = pd.merge(
    df_air_pivot, df_weather_monthly,
    on=['location_name','year','month'],
    how='inner'  # only keep rows where both exist
)
print(df_merged.shape)

(293, 13)


In [39]:
df_tourism = pd.read_csv('nepal_tourism_monthly_2021_2025.csv')

df_final = pd.merge(
    df_merged, df_tourism,
    on=['year','month'],
    how='left'  # every row gets the national tourism number for that month
)
print(df_final.shape)
df_final.to_csv('nepal_air_tourism_climate_merged.csv', index=False)

(293, 14)


In [40]:
print(df_air_pivot['location_name'].nunique())
print(df_weather_monthly['location_name'].nunique())
print(set(df_air_pivot['location_name'].unique()) & set(df_weather_monthly['location_name'].unique()))

70
69
{'CEN-SR-02: Farsidol Brick Factories', 'Gokarneshwor (SC-13) - GD Labs', 'CEN_SR-09: Dhangadhdi Sub-metropolitan City Office', 'CEN-SR-25: Patako Chowk, Patan Durbar Square', 'Lagankhel (SC - 05) - GD Labs', 'Balkumari(SC-28)- GD Labs', 'CEN-SR-03:Biratnagar metropolitan city office ', 'Mahankal (SC-16) - GD Labs', 'Purano naikap (SC-29)-GD Labs', 'Tyanglaphat (SC - 21) - GD Labs', 'Jadibuti (SC-35)-GD Labs', 'Sanepa (SC - 22) - GD Labs', 'Gothatar (SC-12) - GD Labs', 'Sitapaila (SC-30) - GD Labs', 'Taudaha (SC - 09) - GD Labs', 'Golfutar (SC- 17) - GD Labs', 'CEN-SR-16: Pokhara Metropolitan City Ward No. 7 Office', 'CEN-SR-20: Janakpurdham SMC-08, Rajarshi Janak University', 'Balaju (SC-26)- GD Labs', 'Hetauda Udhyog Sang Office (CEN-SR-18)', 'CEN-SR-22:Birendranagar Ward 12 Sahakari Chowk', 'Kaushaltar (SC - 33) GD labs', 'CEN-SR-10: Siddharthanagar Municipality Ward no. 4', 'Ramkot (SC - 10) - GD Labs', 'Tokha (SC - 32) - GD Labs', 'Bharatpur Ward no 27 office Meghauli', 'Nak

In [41]:
# Use weather as the base (it has the most granular data)
df_final = pd.merge(
    df_weather_monthly, df_air_pivot,
    on=['location_name','year','month'],
    how='left'   # <-- keep ALL weather rows, even if no matching air quality
)

df_final = pd.merge(
    df_final, df_tourism,
    on=['year','month'],
    how='left'   # <-- attach national tourism data to every row
)

print(df_final.shape)

(4140, 14)


In [42]:
# Instead of inner join (only matches), use outer or left join
# so you keep every air quality + weather row even without a perfect match,
# with tourism attached where possible
df_merged = pd.merge(
    df_air_pivot, df_weather_monthly,
    on=['location_name','year','month'],
    how='outer'  # keeps everything, fills gaps with NaN
)
print(df_merged.shape)

(4153, 13)


In [43]:
df_merged

,location_name,year,month,o3,pm1,pm10,pm25,relativehumidity,temperature,um003,temperature_c,precipitation_mm,humidity_pct
0,"FHI-OU-08: Amda Hospital, BSMC",2021,Apr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.192667,21.13,18.038000
1,"FHI-OU-08: Amda Hospital, BSMC",2021,Aug,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.609032,700.56,89.846452
2,"FHI-OU-08: Amda Hospital, BSMC",2021,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.030000,39.97,74.434194
3,"FHI-OU-08: Amda Hospital, BSMC",2021,Feb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.857143,3.18,34.830357
4,"FHI-OU-08: Amda Hospital, BSMC",2021,Jan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.673548,0.27,44.063226
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4148,Tyanglaphat (SC - 21) - GD Labs,2025,Mar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.984839,80.61,36.015484
4149,Tyanglaphat (SC - 21) - GD Labs,2025,May,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.783548,166.05,56.850968
4150,Tyanglaphat (SC - 21) - GD Labs,2025,Nov,NaN,28.3,NaN,45.9,49.4,20.2,2000.0,17.908667,7.74,77.712667
4151,Tyanglaphat (SC - 21) - GD Labs,2025,Oct,NaN,25.0,NaN,40.5,50.2,22.6,1650.0,22.176129,238.17,82.687097


In [44]:
# Don't aggregate weather to monthly — use the daily version directly
df_weather_daily = pd.read_csv('nepal_weather_daily_2021_2025.csv')
df_weather_daily['date'] = pd.to_datetime(df_weather_daily['date'].astype(str), format='%Y%m%d')
df_weather_daily['year'] = df_weather_daily['date'].dt.year
df_weather_daily['month'] = df_weather_daily['date'].dt.strftime('%b')

# Join monthly air quality onto daily weather (one-to-many)
df_merged_daily = pd.merge(
    df_weather_daily, df_air_pivot,
    on=['location_name','year','month'],
    how='inner'  # keep only real station matches, no padding
)
print(df_merged_daily.shape)

(8946, 16)


In [45]:
df_merged_daily

,location_name,latitude,longitude,date,temperature_c,precipitation_mm,humidity_pct,year,month,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
0,Embassy Kathmandu,27.738703,85.336206,2021-01-01,13.61,0.00,43.79,2021,Jan,0.0222,NaN,NaN,102.0,NaN,NaN,NaN
1,Embassy Kathmandu,27.738703,85.336206,2021-01-02,13.90,0.00,42.38,2021,Jan,0.0222,NaN,NaN,102.0,NaN,NaN,NaN
2,Embassy Kathmandu,27.738703,85.336206,2021-01-03,15.05,0.00,42.59,2021,Jan,0.0222,NaN,NaN,102.0,NaN,NaN,NaN
3,Embassy Kathmandu,27.738703,85.336206,2021-01-04,16.34,0.00,47.81,2021,Jan,0.0222,NaN,NaN,102.0,NaN,NaN,NaN
4,Embassy Kathmandu,27.738703,85.336206,2021-01-05,16.47,0.00,53.05,2021,Jan,0.0222,NaN,NaN,102.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8941,CEN-SR-19 : Bharatpur Ward 02,27.697650,84.432300,2025-12-27,14.09,0.00,71.92,2025,Dec,NaN,52.2,NaN,89.5,52.6,20.5,3900.0
8942,CEN-SR-19 : Bharatpur Ward 02,27.697650,84.432300,2025-12-28,12.95,0.00,74.49,2025,Dec,NaN,52.2,NaN,89.5,52.6,20.5,3900.0
8943,CEN-SR-19 : Bharatpur Ward 02,27.697650,84.432300,2025-12-29,12.34,0.21,71.78,2025,Dec,NaN,52.2,NaN,89.5,52.6,20.5,3900.0
8944,CEN-SR-19 : Bharatpur Ward 02,27.697650,84.432300,2025-12-30,11.50,0.02,66.73,2025,Dec,NaN,52.2,NaN,89.5,52.6,20.5,3900.0


In [46]:
df_tourism = pd.read_csv('nepal_tourism_monthly_2021_2025.csv')

df_final = pd.merge(
    df_merged_daily, df_tourism,
    on=['year','month'],
    how='left'  # tourism is national, so every row gets a value — no data lost
)
print(df_final.shape)
print(df_final.isna().sum())  # sanity check

(8946, 17)
location_name          0
latitude               0
longitude              0
date                   0
temperature_c          0
precipitation_mm       0
humidity_pct           0
year                   0
month                  0
o3                  6329
pm1                 3468
pm10                8333
pm25                  59
relativehumidity    3468
temperature         3255
um003               3468
tourist_arrivals       0
dtype: int64


In [47]:
print(df_air_pivot['location_name'].nunique(), "air quality stations")
print(df_weather_daily['location_name'].nunique(), "weather stations")
overlap = set(df_air_pivot['location_name'].unique()) & set(df_weather_daily['location_name'].unique())
print(len(overlap), "stations overlap")

70 air quality stations
69 weather stations
62 stations overlap


In [48]:
def get_daily_air_data(sensor_id):
    all_rows = []
    page = 1
    while True:
        resp = requests.get(
            f"{BASE_URL}/sensors/{sensor_id}/days",
            headers=HEADERS,
            params={"datetime_from": "2021-01-01", "datetime_to": "2025-12-31", "limit": 1000, "page": page}
        )
        if resp.status_code == 429:
            time.sleep(10)
            continue
        if resp.status_code != 200:
            return all_rows
        data = resp.json()
        results = data.get("results", [])
        if not results:
            break
        all_rows.extend(results)
        if len(results) < 1000:
            break
        page += 1
    return all_rows

# Only pull for sensors belonging to your 62 overlapping stations
overlap_sensors = [s for s in all_sensors if s["location_name"] in overlap]

daily_air_records = []
for i, s in enumerate(overlap_sensors):
    rows = get_daily_air_data(s["sensor_id"])
    for r in rows:
        r["sensor_id"] = s["sensor_id"]
        r["parameter_name"] = s["parameter"]
        r["location_name"] = s["location_name"]
    daily_air_records.extend(rows)
    print(f"{i}/{len(overlap_sensors)} done, total: {len(daily_air_records)}")
    time.sleep(0.3)

print(f"TOTAL DAILY AIR ROWS: {len(daily_air_records)}")

0/306 done, total: 2320
1/306 done, total: 5396
2/306 done, total: 8048
3/306 done, total: 10871
4/306 done, total: 11130
5/306 done, total: 11370
6/306 done, total: 11796
7/306 done, total: 12055
8/306 done, total: 12481
9/306 done, total: 12740
10/306 done, total: 13381
11/306 done, total: 13620
12/306 done, total: 14261
13/306 done, total: 14902
14/306 done, total: 15543
15/306 done, total: 16184
16/306 done, total: 16392
17/306 done, total: 16600
18/306 done, total: 16808
19/306 done, total: 17016
20/306 done, total: 17224
21/306 done, total: 17378
22/306 done, total: 17532
23/306 done, total: 17686
24/306 done, total: 17840
25/306 done, total: 17994
26/306 done, total: 18245
27/306 done, total: 18496
28/306 done, total: 18747
29/306 done, total: 18998
30/306 done, total: 19249
31/306 done, total: 19381
32/306 done, total: 19513
33/306 done, total: 19645
34/306 done, total: 19777
35/306 done, total: 19909
36/306 done, total: 20008
37/306 done, total: 20107
38/306 done, total: 20206

In [66]:
df_air_daily = pd.DataFrame(daily_air_records)

In [67]:
print(df_air_daily.columns.tolist())

['value', 'flagInfo', 'parameter', 'period', 'coordinates', 'summary', 'coverage', 'sensor_id', 'parameter_name', 'location_name']


In [68]:
print(df_air_daily.head())

   value             flagInfo  \
0  0.029  {'hasFlags': False}   
1 -0.746   {'hasFlags': True}   
2 -0.999   {'hasFlags': True}   
3 -0.999   {'hasFlags': True}   
4 -0.999   {'hasFlags': True}   

                                           parameter  \
0  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   
1  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   
2  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   
3  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   
4  {'id': 10, 'name': 'o3', 'units': 'ppm', 'disp...   

                                              period coordinates  \
0  {'label': '1day', 'interval': '24:00:00', 'dat...        None   
1  {'label': '1day', 'interval': '24:00:00', 'dat...        None   
2  {'label': '1day', 'interval': '24:00:00', 'dat...        None   
3  {'label': '1day', 'interval': '24:00:00', 'dat...        None   
4  {'label': '1day', 'interval': '24:00:00', 'dat...        None   

                                             summary  \

In [52]:
print(df_air_daily["period"].iloc[0])

{'label': '1day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2017-03-02T18:15:00Z', 'local': '2017-03-03T00:00:00+05:45'}, 'datetimeTo': {'utc': '2017-03-03T18:15:00Z', 'local': '2017-03-04T00:00:00+05:45'}}


In [53]:
df_air_daily["date"] = pd.to_datetime(
    df_air_daily["period"].apply(lambda x: x["datetimeFrom"]["local"])
).dt.date

In [54]:
print(df_air_daily[["location_name", "date"]].head())

       location_name        date
0  Embassy Kathmandu  2017-03-03
1  Embassy Kathmandu  2017-03-04
2  Embassy Kathmandu  2017-03-05
3  Embassy Kathmandu  2017-03-06
4  Embassy Kathmandu  2017-03-07


In [55]:
df_air_daily_pivot = df_air_daily.pivot_table(
    index=["location_name", "date"],
    columns="parameter_name",
    values="value",
    aggfunc="mean"
).reset_index()

print(df_air_daily_pivot.shape)

(17127, 9)


In [56]:
print(df_air_daily_pivot.shape)

(17127, 9)


In [57]:
df_air_daily_pivot["date"] = pd.to_datetime(df_air_daily_pivot["date"])
df_weather_daily["date"] = pd.to_datetime(df_weather_daily["date"])

df_merged_daily2 = pd.merge(
    df_weather_daily,
    df_air_daily_pivot,
    on=["location_name", "date"],
    how="inner"
)

print(df_merged_daily2.shape)

(7155, 16)


In [58]:
df_merged_daily2.head()

,location_name,latitude,longitude,date,temperature_c,precipitation_mm,humidity_pct,year,month,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
0,Embassy Kathmandu,27.738703,85.336206,2021-01-01,13.61,0.0,43.79,2021,Jan,0.0194,NaN,NaN,109.0,NaN,NaN,NaN
1,Embassy Kathmandu,27.738703,85.336206,2021-01-02,13.90,0.0,42.38,2021,Jan,0.0032,NaN,NaN,131.0,NaN,NaN,NaN
2,Embassy Kathmandu,27.738703,85.336206,2021-01-03,15.05,0.0,42.59,2021,Jan,0.0277,NaN,NaN,105.0,NaN,NaN,NaN
3,Embassy Kathmandu,27.738703,85.336206,2021-01-04,16.34,0.0,47.81,2021,Jan,0.0119,NaN,NaN,265.0,NaN,NaN,NaN
4,Embassy Kathmandu,27.738703,85.336206,2021-01-05,16.47,0.0,53.05,2021,Jan,0.0162,NaN,NaN,258.0,NaN,NaN,NaN


In [59]:
print("Weather stations:", df_weather_daily["location_name"].nunique())
print("Air stations:", df_air_daily_pivot["location_name"].nunique())

print("Weather date range:",
      df_weather_daily["date"].min(),
      df_weather_daily["date"].max())

print("Air date range:",
      df_air_daily_pivot["date"].min(),
      df_air_daily_pivot["date"].max())

Weather stations: 69
Air stations: 62
Weather date range: 2021-01-01 00:00:00 2025-12-31 00:00:00
Air date range: 2017-03-03 00:00:00 2026-07-13 00:00:00


In [60]:
common = set(df_weather_daily["location_name"]) & set(df_air_daily_pivot["location_name"])

print("Common stations:", len(common))
print(sorted(common))

Common stations: 62
['Balaju (SC-26)- GD Labs', 'Balkumari(SC-28)- GD Labs', 'Baluwatar (SC-02) - GD Labs', 'Bharatpur Ward no 27 office Meghauli', 'CEN-SR-02: Farsidol Brick Factories', 'CEN-SR-03:Biratnagar metropolitan city office ', 'CEN-SR-04: Birgunj Metropolitan City Ward No 25', 'CEN-SR-05: Shree Vindhyavasini Secondary School, Pokhara', 'CEN-SR-06: Biratnagar Metropolitan City ward no 9 Office', 'CEN-SR-07: Siddharthanagar Municipality City Office Ward No. 5', 'CEN-SR-08/ Birendranagar Municipality', 'CEN-SR-10: Siddharthanagar Municipality Ward no. 4', 'CEN-SR-11: Janakpurdham Sub-metropolitian City Office', 'CEN-SR-12: Lamahi Municipality Office, Dang', 'CEN-SR-13: Birgunj Metropolitan City Office', 'CEN-SR-14: Dhangadhi Sub-metropolitan Ward 8', 'CEN-SR-16: Pokhara Metropolitan City Ward No. 7 Office', 'CEN-SR-17: Bhaluwang, Dang', 'CEN-SR-19 : Bharatpur Ward 02', 'CEN-SR-20: Janakpurdham SMC-08, Rajarshi Janak University', 'CEN-SR-22:Birendranagar Ward 12 Sahakari Chowk', 

In [61]:
df_merged_daily2 = pd.merge(
    df_weather_daily,
    df_air_daily_pivot,
    on=["location_name", "date"],
    how="left"
)

In [62]:
df_merged_daily2 

,location_name,latitude,longitude,date,temperature_c,precipitation_mm,humidity_pct,year,month,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
0,Embassy Kathmandu,27.738703,85.336206,2021-01-01,13.61,0.00,43.79,2021,Jan,0.0194,NaN,NaN,109.0,NaN,NaN,NaN
1,Embassy Kathmandu,27.738703,85.336206,2021-01-02,13.90,0.00,42.38,2021,Jan,0.0032,NaN,NaN,131.0,NaN,NaN,NaN
2,Embassy Kathmandu,27.738703,85.336206,2021-01-03,15.05,0.00,42.59,2021,Jan,0.0277,NaN,NaN,105.0,NaN,NaN,NaN
3,Embassy Kathmandu,27.738703,85.336206,2021-01-04,16.34,0.00,47.81,2021,Jan,0.0119,NaN,NaN,265.0,NaN,NaN,NaN
4,Embassy Kathmandu,27.738703,85.336206,2021-01-05,16.47,0.00,53.05,2021,Jan,0.0162,NaN,NaN,258.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125989,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-27,14.28,0.00,65.19,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125990,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-28,13.75,0.00,64.38,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125991,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-29,13.04,0.32,63.74,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125992,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-30,12.38,0.04,60.46,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
print(df_merged_daily2["pm25"].notna().sum())
print(df_merged_daily2["o3"].notna().sum())

6659
2262


In [64]:
print(df_merged_daily2.isna().sum())

location_name            0
latitude                 0
longitude                0
date                     0
temperature_c            0
precipitation_mm         0
humidity_pct             0
year                     0
month                    0
o3                  123732
pm1                 122004
pm10                125515
pm25                119335
relativehumidity    122004
temperature         121837
um003               122004
dtype: int64


In [69]:
print(df_air_daily["parameter_name"].value_counts())

parameter_name
pm25                17246
temperature         11354
relativehumidity    11187
um003               11187
pm1                 11180
o3                   4972
pm10                  479
Name: count, dtype: int64


In [70]:
print(df_air_daily_pivot.columns.tolist())

['location_name', 'date', 'o3', 'pm1', 'pm10', 'pm25', 'relativehumidity', 'temperature', 'um003']


In [71]:
print(df_air_daily["location_name"].nunique())
print(df_air_daily["sensor_id"].nunique())

62
306


In [72]:
print(df_air_daily["parameter_name"].value_counts())

print(df_air_daily_pivot.columns.tolist())

print(df_air_daily["location_name"].nunique())

print(df_air_daily["sensor_id"].nunique())

parameter_name
pm25                17246
temperature         11354
relativehumidity    11187
um003               11187
pm1                 11180
o3                   4972
pm10                  479
Name: count, dtype: int64
['location_name', 'date', 'o3', 'pm1', 'pm10', 'pm25', 'relativehumidity', 'temperature', 'um003']
62
306


In [73]:
print(df_tourism.columns.tolist())
print(df_tourism.head())

['year', 'month', 'tourist_arrivals']
   year month  tourist_arrivals
0  2021   Jan              8874
1  2021   Feb              9266
2  2021   Mar             15254
3  2021   Apr             22732
4  2021   May              1531


In [74]:
print(df_final["tourist_arrivals"].isna().sum())

0


In [75]:
df_merged = pd.merge(
    df_weather_daily,
    df_air_daily_pivot,
    on=["location_name", "date"],
    how="left"
)

In [76]:
df_final = pd.merge(
    df_merged,
    df_tourism,
    on=["year", "month"],
    how="left"
)

In [77]:
print(df_final.shape)

(125994, 17)


In [78]:
df_final

,location_name,latitude,longitude,date,temperature_c,precipitation_mm,humidity_pct,year,month,o3,pm1,pm10,pm25,relativehumidity,temperature,um003,tourist_arrivals
0,Embassy Kathmandu,27.738703,85.336206,2021-01-01,13.61,0.00,43.79,2021,Jan,0.0194,NaN,NaN,109.0,NaN,NaN,NaN,8874
1,Embassy Kathmandu,27.738703,85.336206,2021-01-02,13.90,0.00,42.38,2021,Jan,0.0032,NaN,NaN,131.0,NaN,NaN,NaN,8874
2,Embassy Kathmandu,27.738703,85.336206,2021-01-03,15.05,0.00,42.59,2021,Jan,0.0277,NaN,NaN,105.0,NaN,NaN,NaN,8874
3,Embassy Kathmandu,27.738703,85.336206,2021-01-04,16.34,0.00,47.81,2021,Jan,0.0119,NaN,NaN,265.0,NaN,NaN,NaN,8874
4,Embassy Kathmandu,27.738703,85.336206,2021-01-05,16.47,0.00,53.05,2021,Jan,0.0162,NaN,NaN,258.0,NaN,NaN,NaN,8874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125989,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-27,14.28,0.00,65.19,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98180
125990,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-28,13.75,0.00,64.38,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98180
125991,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-29,13.04,0.32,63.74,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98180
125992,"FHI-OU-05: Nayagaun Secondary School, BSMC",27.663450,83.409495,2025-12-30,12.38,0.04,60.46,2025,Dec,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98180


In [79]:
df_final.to_csv("nepal_climate_air_tourism.csv", index=False)

# FINAL DATASET SOLVE

In [1]:
import requests
import pandas as pd
import time

In [2]:
API_KEY = "32f25e47582fbae77e1c5ba484f1c6db85bfa267c03e619d91487b66d98d0c30"
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"
COUNTRY_ID = 145  

In [3]:
#Getting all Nepal Stations
def get_nepal_locations():
    locations = []
    page = 1
    while True:
        resp = requests.get(f"{BASE_URL}/locations", headers=HEADERS,
                             params={"countries_id": COUNTRY_ID, "limit": 1000, "page": page})
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            break
        locations.extend(results)
        if len(results) < 1000:
            break
        page += 1
        time.sleep(0.5)
    return locations

nepal_locations = get_nepal_locations()
print(f"Found {len(nepal_locations)} stations")

Found 84 stations


In [4]:
#Getting all sensors
def get_all_sensors(locations):
    sensor_list = []
    for loc in locations:
        for s in loc.get("sensors", []):
            sensor_list.append({
                "sensor_id": s["id"],
                "parameter": s["parameter"]["name"],
                "location_id": loc["id"],
                "location_name": loc["name"]
            })
    return sensor_list

all_sensors = get_all_sensors(nepal_locations)
print(f"Total sensors: {len(all_sensors)}")

Total sensors: 416


In [5]:
# Pulling daily air quality data (2021-2025)
def get_daily_air_data(sensor_id):
    all_rows = []
    page = 1
    while True:
        resp = requests.get(f"{BASE_URL}/sensors/{sensor_id}/days", headers=HEADERS,
                             params={"datetime_from": "2021-01-01", "datetime_to": "2025-12-31",
                                     "limit": 1000, "page": page})
        if resp.status_code == 429:
            time.sleep(10); continue
        if resp.status_code != 200:
            return all_rows
        results = resp.json().get("results", [])
        if not results:
            break
        all_rows.extend(results)
        if len(results) < 1000:
            break
        page += 1
    return all_rows

air_records = []
for i, s in enumerate(all_sensors):
    rows = get_daily_air_data(s["sensor_id"])
    for r in rows:
        r["sensor_id"] = s["sensor_id"]
        r["parameter_name"] = s["parameter"]
        r["location_name"] = s["location_name"]
    air_records.extend(rows)
    if i % 20 == 0:
        print(f"{i}/{len(all_sensors)} sensors, total rows: {len(air_records)}")
    time.sleep(0.3)

print(f"TOTAL AIR ROWS: {len(air_records)}")

0/416 sensors, total rows: 2320
20/416 sensors, total rows: 17274
40/416 sensors, total rows: 20764
60/416 sensors, total rows: 24234
80/416 sensors, total rows: 27494
100/416 sensors, total rows: 31409
120/416 sensors, total rows: 35204
140/416 sensors, total rows: 39279
160/416 sensors, total rows: 43829
180/416 sensors, total rows: 47864
200/416 sensors, total rows: 51394
220/416 sensors, total rows: 55710
240/416 sensors, total rows: 59165
260/416 sensors, total rows: 62275
280/416 sensors, total rows: 65330
300/416 sensors, total rows: 68555
320/416 sensors, total rows: 71280
340/416 sensors, total rows: 72910
360/416 sensors, total rows: 73600
380/416 sensors, total rows: 74536
400/416 sensors, total rows: 74921
TOTAL AIR ROWS: 74971


In [6]:
#Cleaning air quality into a pivoted table (one row per station-day)
df_air = pd.DataFrame(air_records)
df_air["date"] = pd.to_datetime(df_air["period"].apply(lambda x: x["datetimeFrom"]["local"])).dt.date
df_air["date"] = pd.to_datetime(df_air["date"])

df_air_pivot = df_air.pivot_table(
    index=["location_name","date"],
    columns="parameter_name",
    values="value",
    aggfunc="mean"
).reset_index()

print(df_air_pivot.shape)

(18599, 9)


In [7]:
df_air_pivot

parameter_name,location_name,date,o3,pm1,pm10,pm25,relativehumidity,temperature,um003
0,"FHI-OU-08: Amda Hospital, BSMC",2026-06-20,NaN,35.500,NaN,56.80,57.5,32.1,2900.0
1,"FHI-OU-08: Amda Hospital, BSMC",2026-06-21,NaN,39.800,NaN,62.90,53.4,33.8,3210.0
2,"FHI-OU-08: Amda Hospital, BSMC",2026-06-22,NaN,34.900,NaN,54.50,52.4,33.5,2760.0
3,"FHI-OU-08: Amda Hospital, BSMC",2026-06-23,NaN,27.600,NaN,42.60,52.3,32.3,2000.0
4,"FHI-OU-08: Amda Hospital, BSMC",2026-06-24,NaN,25.900,NaN,39.00,44.9,34.3,1760.0
...,...,...,...,...,...,...,...,...,...
18594,Tyanglaphat (SC - 21) - GD Labs,2025-10-29,NaN,39.500,NaN,63.90,54.4,21.4,2960.0
18595,Tyanglaphat (SC - 21) - GD Labs,2025-10-30,NaN,24.000,NaN,38.70,62.1,20.6,1660.0
18596,Tyanglaphat (SC - 21) - GD Labs,2025-10-31,NaN,11.200,NaN,17.10,63.7,21.3,747.0
18597,Tyanglaphat (SC - 21) - GD Labs,2025-11-01,NaN,0.931,NaN,2.09,63.6,21.6,259.0


In [12]:
df_air_pivot.to_csv("grdv_air_quality_daily_clean.csv", index=False)

In [8]:
#Pulling weather (NASA POWER, daily, 2021–2025) per unique station location
def get_nasa_power_data(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {"parameters": "T2M,PRECTOTCORR,RH2M", "community": "AG",
              "longitude": lon, "latitude": lat,
              "start": "20210101", "end": "20251231", "format": "JSON"}
    resp = requests.get(url, params=params)
    if resp.status_code != 200:
        return None
    return resp.json()

weather_records = []
seen_coords = set()
for loc in nepal_locations:
    coords = loc.get("coordinates")
    if not coords:
        continue
    lat, lon = coords["latitude"], coords["longitude"]
    key = (round(lat,2), round(lon,2))
    if key in seen_coords:
        continue
    seen_coords.add(key)

    data = get_nasa_power_data(lat, lon)
    if data is None:
        continue
    params_data = data.get("properties", {}).get("parameter", {})
    for date in params_data.get("T2M", {}):
        weather_records.append({
            "location_name": loc["name"],
            "date": date,
            "temperature_c": params_data.get("T2M", {}).get(date),
            "precipitation_mm": params_data.get("PRECTOTCORR", {}).get(date),
            "humidity_pct": params_data.get("RH2M", {}).get(date),
        })
    time.sleep(1)

df_weather = pd.DataFrame(weather_records)
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y%m%d")
print(df_weather.shape)

(129646, 5)


In [13]:
df_weather.to_csv("grdv_weather_daily_clean.csv", index=False)

In [9]:
import pypdf

reader = pypdf.PdfReader('Nepal tourism statistics 2025_ibddtm6.pdf')

# Table 2.2 spans pages 29 and 30 (0-indexed)
page1_text = reader.pages[29].extract_text()
page2_text = reader.pages[30].extract_text()

print(page1_text)
print(page2_text)

12 | NEPAL TOURISM STATISTICS, 2025
TABLE 2.2: TOURIST ARRIVAL BY MONTH, 1995-2025
Y ear Jan. Feb. Mar. Apr. May Jun. Jul. Aug. Sep. Oct. Nov. Dec. Total
1995 22,207 28,240 34,219 33,994 27,843 25,650 23,980 27,686 30,569 46,845 35,782 26,380 363,395
1996 27,886 29,676 39,336 36,331 29,728 26,749 22,684 29,080 32,181 47,314 37,650 34,998 393,613
1997 25,585 32,861 43,177 35,229 33,456 26,367 26,091 35,549 31,981 56,272 40,173 35,116 421,857
1998 28,822 37,956 41,338 41,087 35,814 29,181 27,895 36,174 39,664 62,487 47,403 35,863 463,684
1999 29,752 38,134 46,218 40,774 42,712 31,049 27,193 38,449 44,117 66,543 48,865 37,698 491,504
2000 25,307 38,959 44,944 43,635 28,363 26,933 24,480 34,670 43,523 59,195 52,993 40,644 463,646
2001 30,454 38,680 46,709 39,083 28,345 13,030 18,329 25,322 31,170 41,245 30,282 18,588 361,237
2002 17,176 20,668 28,815 21,253 19,887 17,218 16,621 21,093 23,752 35,272 28,723 24,990 275,468
2003 21,215 24,349 27,737 25,851 22,704 20,351 22,661 27,568 28,724 45

In [10]:
raw_tourism = """2021 8,874 9,266 15,254 22,732 1,531 1,187 3,093 6,093 9,907 23,338 26,135 23,552 150,962
2022 16975 19856 42152 61589 54093 46957 44462 41304 58314 88582 72653 67932 614869
2023 55074 73255 99426 98773 77703 72250 57726 67153 91012 117306 108630 96574 1014882
2024 79101 97423 128167 111382 90205 76733 64598 72717 96302 124391 114496 92033 1147548
2025 79990 99585 121632 116487 86216 76425 70193 88681 79981 128443 116552 98180 1162365"""

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
records = []
for line in raw_tourism.strip().split('\n'):
    parts = line.split()
    year = int(parts[0])
    nums = [int(p.replace(',','')) for p in parts[1:13]]
    for m, val in zip(months, nums):
        records.append({'year': year, 'month': m, 'tourist_arrivals': val})

df_tourism = pd.DataFrame(records)

In [11]:
df_tourism

,year,month,tourist_arrivals
0,2021,Jan,8874
1,2021,Feb,9266
2,2021,Mar,15254
3,2021,Apr,22732
4,2021,May,1531
5,2021,Jun,1187
6,2021,Jul,3093
7,2021,Aug,6093
8,2021,Sep,9907
9,2021,Oct,23338


In [14]:
df_tourism.to_csv("grdv_tourism_monthly_clean.csv", index=False)

In [15]:
#Final merge (daily air + daily weather, inner join; then tourism, left join)
df_merged = pd.merge(
    df_weather, df_air_pivot,
    on=["location_name","date"], how="inner"
)

df_merged["year"] = df_merged["date"].dt.year
df_merged["month"] = df_merged["date"].dt.strftime('%b')

df_final = pd.merge(
    df_merged, df_tourism,
    on=["year","month"], how="left"
)

print(df_final.shape)

(7155, 15)


In [16]:
df_final

,location_name,date,temperature_c,precipitation_mm,humidity_pct,o3,pm1,pm10,pm25,relativehumidity,temperature,um003,year,month,tourist_arrivals
0,Embassy Kathmandu,2021-01-01,13.61,0.00,43.79,0.0194,NaN,NaN,109.0,NaN,NaN,NaN,2021,Jan,8874
1,Embassy Kathmandu,2021-01-02,13.90,0.00,42.38,0.0032,NaN,NaN,131.0,NaN,NaN,NaN,2021,Jan,8874
2,Embassy Kathmandu,2021-01-03,15.05,0.00,42.59,0.0277,NaN,NaN,105.0,NaN,NaN,NaN,2021,Jan,8874
3,Embassy Kathmandu,2021-01-04,16.34,0.00,47.81,0.0119,NaN,NaN,265.0,NaN,NaN,NaN,2021,Jan,8874
4,Embassy Kathmandu,2021-01-05,16.47,0.00,53.05,0.0162,NaN,NaN,258.0,NaN,NaN,NaN,2021,Jan,8874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7150,CEN-SR-19 : Bharatpur Ward 02,2025-12-27,14.09,0.00,71.92,NaN,72.5,NaN,134.0,57.7,18.8,5540.0,2025,Dec,98180
7151,CEN-SR-19 : Bharatpur Ward 02,2025-12-28,12.95,0.00,74.49,NaN,43.8,NaN,80.3,57.4,18.9,3380.0,2025,Dec,98180
7152,CEN-SR-19 : Bharatpur Ward 02,2025-12-29,12.34,0.21,71.78,NaN,41.6,NaN,72.2,57.2,18.6,3090.0,2025,Dec,98180
7153,CEN-SR-19 : Bharatpur Ward 02,2025-12-30,11.50,0.02,66.73,NaN,37.5,NaN,63.0,55.5,18.3,2790.0,2025,Dec,98180


In [27]:
df_final.to_csv("grdv_nepal_t1_dataset.csv", index=False)

In [17]:
print(df_final.groupby('month')['precipitation_mm'].mean())
print(df_final['pm25'].notna().sum(), "rows have pm25")
print(df_final['pm1'].notna().sum(), "rows have pm1")

month
Apr     0.909234
Aug     9.622444
Dec     0.039013
Feb     0.370706
Jan     0.040085
Jul    11.697000
Jun     8.773589
Mar     2.994728
May     4.177445
Nov     0.128091
Oct     3.925206
Sep     6.567704
Name: precipitation_mm, dtype: float64
6659 rows have pm25
3990 rows have pm1


In [18]:
# Does low rainfall correlate with high PM2.5? (your core climate→environment claim)
monthly_avg = df_final.groupby('month').agg(
    avg_pm25=('pm25','mean'),
    avg_precip=('precipitation_mm','mean')
).reindex(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
print(monthly_avg)

        avg_pm25  avg_precip
month                       
Jan    87.656442    0.040085
Feb    81.171489    0.370706
Mar    81.184279    2.994728
Apr    90.799115    0.909234
May    41.338498    4.177445
Jun    28.399505    8.773589
Jul    16.882857   11.697000
Aug    15.782603    9.622444
Sep    20.264563    6.567704
Oct    37.546470    3.925206
Nov    64.567945    0.128091
Dec    93.858333    0.039013


In [19]:
#Check for impossible/negative values (sensor errors)
print(df_final[['temperature_c','precipitation_mm','humidity_pct','pm25','pm1','o3']].describe())

       temperature_c  precipitation_mm  humidity_pct         pm25  \
count    7155.000000       7155.000000   7155.000000  6659.000000   
mean       17.693898          2.695680     71.571948    62.399943   
std         5.046883          9.034856     16.193257    48.278787   
min         1.630000          0.000000     16.320000     0.000000   
25%        13.600000          0.000000     63.560000    31.100000   
50%        17.440000          0.000000     74.540000    53.500000   
75%        22.210000          0.725000     83.960000    80.700000   
max        32.050000        233.660000     97.140000   438.000000   

               pm1           o3  
count  3990.000000  2262.000000  
mean     41.067154    -0.439045  
std      26.985869     0.468027  
min       0.000000    -0.999000  
25%      23.900000    -0.999000  
50%      35.700000    -0.216000  
75%      50.900000     0.017800  
max     197.000000     0.075900  


In [20]:
print((df_final['pm25'] == 0).sum())

1


In [22]:
print(df_final.isna().sum())

location_name          0
date                   0
temperature_c          0
precipitation_mm       0
humidity_pct           0
o3                  4893
pm1                 3165
pm10                6676
pm25                 496
relativehumidity    3165
temperature         2998
um003               3165
year                   0
month                  0
tourist_arrivals       0
dtype: int64


In [23]:
print(df_final.isna().mean().round(3) * 100)  # percentage missing per column

location_name        0.0
date                 0.0
temperature_c        0.0
precipitation_mm     0.0
humidity_pct         0.0
o3                  68.4
pm1                 44.2
pm10                93.3
pm25                 6.9
relativehumidity    44.2
temperature         41.9
um003               44.2
year                 0.0
month                0.0
tourist_arrivals     0.0
dtype: float64


In [24]:
df_final_clean = df_final[['location_name','date','year','month',
                             'temperature_c','precipitation_mm','humidity_pct',
                             'pm25','pm1','tourist_arrivals']]
df_final_clean = df_final_clean.rename(columns={
    'temperature_c':'temperature',
    'precipitation_mm':'rainfall',
    'humidity_pct':'humidity'
})
print(df_final_clean.shape)
print(df_final_clean.isna().sum())

(7155, 10)
location_name          0
date                   0
year                   0
month                  0
temperature            0
rainfall               0
humidity               0
pm25                 496
pm1                 3165
tourist_arrivals       0
dtype: int64


In [25]:
print("Exact duplicates:", df_final_clean.duplicated().sum())
print("Duplicate station+date:", df_final_clean.duplicated(subset=['location_name','date']).sum())

for col in ['temperature','rainfall','humidity','pm25','pm1']:
    neg = (df_final_clean[col] < 0).sum()
    print(f"{col}: {neg} negative values")

print("Humidity below 0:", (df_final_clean['humidity'] < 0).sum())
print("Humidity above 100:", (df_final_clean['humidity'] > 100).sum())

Exact duplicates: 0
Duplicate station+date: 0
temperature: 0 negative values
rainfall: 0 negative values
humidity: 0 negative values
pm25: 0 negative values
pm1: 0 negative values
Humidity below 0: 0
Humidity above 100: 0


In [26]:
df_final_clean

,location_name,date,year,month,temperature,rainfall,humidity,pm25,pm1,tourist_arrivals
0,Embassy Kathmandu,2021-01-01,2021,Jan,13.61,0.00,43.79,109.0,NaN,8874
1,Embassy Kathmandu,2021-01-02,2021,Jan,13.90,0.00,42.38,131.0,NaN,8874
2,Embassy Kathmandu,2021-01-03,2021,Jan,15.05,0.00,42.59,105.0,NaN,8874
3,Embassy Kathmandu,2021-01-04,2021,Jan,16.34,0.00,47.81,265.0,NaN,8874
4,Embassy Kathmandu,2021-01-05,2021,Jan,16.47,0.00,53.05,258.0,NaN,8874
...,...,...,...,...,...,...,...,...,...,...
7150,CEN-SR-19 : Bharatpur Ward 02,2025-12-27,2025,Dec,14.09,0.00,71.92,134.0,72.5,98180
7151,CEN-SR-19 : Bharatpur Ward 02,2025-12-28,2025,Dec,12.95,0.00,74.49,80.3,43.8,98180
7152,CEN-SR-19 : Bharatpur Ward 02,2025-12-29,2025,Dec,12.34,0.21,71.78,72.2,41.6,98180
7153,CEN-SR-19 : Bharatpur Ward 02,2025-12-30,2025,Dec,11.50,0.02,66.73,63.0,37.5,98180


In [28]:
df_final_clean.to_csv('grdv_nepal_t1_dataset_cleaned.csv', index=False)

In [29]:
print("Saved:", df_final_clean.shape)

Saved: (7155, 10)


We tested both left/outer and inner join strategies when merging our air quality and weather datasets. The outer join preserved a larger row count (125,994) but resulted in 94.7% missingness in our primary pollutant variable (PM2.5), as it retained weather records from stations and dates without corresponding air quality readings. We selected an inner join instead, yielding 7,155 rows with 93.1% PM2.5 completeness, prioritizing analytical validity over raw row count — a smaller dataset where every record is genuinely usable outperforms a larger dataset dominated by missing values for our core research question

In [6]:
coords = []
for loc in nepal_locations:
    if loc.get("coordinates"):
        coords.append({
            "location_name": loc["name"],
            "latitude": loc["coordinates"]["latitude"],
            "longitude": loc["coordinates"]["longitude"]
        })

df_coords = pd.DataFrame(coords)
print(df_coords.shape)

(86, 3)


In [7]:
df_coords

,location_name,latitude,longitude
0,Embassy Kathmandu,27.738703,85.336206
1,Phora Durbar Kathman,27.712464,85.315703
2,"Dhathutole, Handigaun",27.727502,85.330135
3,"Dabali, Handigaun",27.681891,85.287078
4,Gaushala Chowk (SC-01) - GD Labs,27.707763,85.343189
...,...,...,...
81,Sainamaina Municipality Ward No. 10,27.702036,83.262739
82,Kalika Manavgyan Secondary School,27.681773,83.465707
83,"FHI-OU-06: Semlar Health Post, BSMC",27.663450,83.409495
84,FHI-OU-01: Butwal Sub-metro City Office,27.707793,83.464583


In [1]:
# Data Validation and Statistical Analysis

In [ ]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent

output_path = project_root / "data" / "processed" / "station_coordinates.csv"
df_coords.to_csv(output_path, index=False)
print(output_path)


In [11]:
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "data" / "processed").exists():
    project_root = project_root.parent

csv_path = project_root / "data" / "processed" / "grdv_nepal_t1_dataset_cleaned.csv"
df = pd.read_csv(csv_path)
print(csv_path)
print('Bagdol' in df['location_name'].unique())


c:\Users\Asus\OneDrive\Desktop\Group DV T1\T1-AirQualityTourism\data\processed\grdv_nepal_t1_dataset_cleaned.csv
False


In [12]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
from scipy import stats
import numpy as np

df = pd.read_csv('../data/processed/grdv_nepal_t1_dataset_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

# Add season column
df['season'] = df['date'].dt.month.apply(lambda m: 'Monsoon' if m in [6,7,8,9] else 'Dry Season')

In [17]:
# --- 1. Overall correlation: Rainfall vs PM2.5 ---
df_clean = df.dropna(subset=['pm25', 'rainfall'])
overall_corr, overall_p = stats.pearsonr(df_clean['rainfall'], df_clean['pm25'])
print(f"Overall correlation (Rainfall vs PM2.5): r = {overall_corr:.3f}, p = {overall_p:.4f}")

# --- 2. Correlation split by season ---
for season in ['Dry Season', 'Monsoon']:
    subset = df_clean[df_clean['season'] == season]
    r, p = stats.pearsonr(subset['rainfall'], subset['pm25'])
    print(f"{season}: r = {r:.3f}, p = {p:.4f}, n = {len(subset)}")

# --- 3. Descriptive stats per season ---
print("\nDescriptive stats by season (PM2.5):")
print(df_clean.groupby('season')['pm25'].agg(['mean','median','std','min','max']).round(2))

# --- 4. % of days exceeding WHO guideline (15 µg/m³) ---
df_clean['exceeds_who'] = df_clean['pm25'] > 15
pct_exceed = df_clean.groupby(df_clean['date'].dt.strftime('%b'))['exceeds_who'].mean() * 100
print("\n% of days exceeding WHO guideline (15 µg/m³) by month:")
print(pct_exceed.round(1))

# --- 5. Correlation: PM2.5 vs Tourist Arrivals (monthly aggregated) ---
monthly = df.groupby(['year','month']).agg(
    avg_pm25=('pm25','mean'),
    tourist_arrivals=('tourist_arrivals','first')
).reset_index().dropna()
tourism_corr, tourism_p = stats.pearsonr(monthly['avg_pm25'], monthly['tourist_arrivals'])
print(f"\nCorrelation (PM2.5 vs Tourist Arrivals, monthly): r = {tourism_corr:.3f}, p = {tourism_p:.4f}")

Overall correlation (Rainfall vs PM2.5): r = -0.234, p = 0.0000
Dry Season: r = -0.147, p = 0.0000, n = 5407
Monsoon: r = -0.035, p = 0.2153, n = 1252

Descriptive stats by season (PM2.5):
             mean  median    std   min    max
season                                       
Dry Season  72.25    63.2  48.20  0.00  438.0
Monsoon     19.86    17.5  11.56  0.31   93.8

% of days exceeding WHO guideline (15 µg/m³) by month:
date
Apr     99.6
Aug     49.2
Dec    100.0
Feb    100.0
Jan    100.0
Jul     53.0
Jun     76.1
Mar     99.6
May     92.0
Nov     97.7
Oct     87.6
Sep     68.6
Name: exceeds_who, dtype: float64

Correlation (PM2.5 vs Tourist Arrivals, monthly): r = 0.127, p = 0.3359


In [15]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\Asus\OneDrive\Desktop\Group DV T1\T1-AirQualityTourism\notebooks
['final.ipynb']


In [18]:
from scipy import stats

# Daily-level correlation (what you already have)
daily_r, daily_p = stats.pearsonr(df_clean['rainfall'], df_clean['pm25'])
print(f"Daily-level correlation: r = {daily_r:.3f}, p = {daily_p:.4f}, n = {len(df_clean)}")

# Monthly-aggregated correlation (average rainfall and PM2.5 per station-month)
monthly_agg = df_clean.groupby(['location_name','year','month']).agg(
    avg_rainfall=('rainfall','mean'),
    avg_pm25=('pm25','mean')
).reset_index()

monthly_r, monthly_p = stats.pearsonr(monthly_agg['avg_rainfall'], monthly_agg['avg_pm25'])
print(f"Monthly-aggregated correlation: r = {monthly_r:.3f}, p = {monthly_p:.4f}, n = {len(monthly_agg)}")

Daily-level correlation: r = -0.234, p = 0.0000, n = 6659
Monthly-aggregated correlation: r = -0.386, p = 0.0000, n = 291


In [19]:
import pandas as pd
from scipy import stats

# Per-station correlation between rainfall and PM2.5
station_correlations = []
for station in df_clean['location_name'].unique():
    subset = df_clean[df_clean['location_name'] == station]
    if len(subset) >= 30:  # only stations with enough data points
        r, p = stats.pearsonr(subset['rainfall'], subset['pm25'])
        station_correlations.append({
            'station': station,
            'r': r,
            'p': p,
            'n': len(subset)
        })

df_station_corr = pd.DataFrame(station_correlations).sort_values('r')
print(df_station_corr.to_string(index=False))

# Check: how many stations show POSITIVE correlation (opposite of overall -0.234)?
positive_stations = df_station_corr[df_station_corr['r'] > 0]
print(f"\nStations with POSITIVE rainfall-PM2.5 correlation (opposite of overall trend): {len(positive_stations)} out of {len(df_station_corr)}")

                                                       station         r            p    n
                                   Taudaha (SC - 09) - GD Labs -0.437029 5.404820e-03   39
                          Bharatpur Ward no 27 office Meghauli -0.421371 1.722463e-05   97
                  CEN-SR-25: Patako Chowk, Patan Durbar Square -0.370736 2.015757e-02   39
                        Hetauda Udhyog Sang Office (CEN-SR-18) -0.353566 5.861796e-04   91
                                   Baluwatar (SC-02) - GD Labs -0.352583 2.564165e-03   71
                                 Lagankhel (SC - 05) - GD Labs -0.351624 7.813411e-04   88
                                    Chovar (SC - 07) - GD Labs -0.351066 2.582583e-04  104
                                 Sunakothi (SC - 06) - GD Labs -0.347601 2.998563e-04  104
CEN-SR-07: Siddharthanagar Municipality City Office Ward No. 5 -0.346134 1.484160e-02   49
                                    Mahankal (SC-16) - GD Labs -0.330905 5.847099e-03   68

C:\Users\Asus\AppData\Local\Temp\ipykernel_25368\4240549752.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = stats.pearsonr(subset['rainfall'], subset['pm25'])


In [20]:
print(df_station_corr[df_station_corr['r'] > 0].to_string(index=False))

                                     station        r        p  n
CEN-SR-14: Dhangadhi Sub-metropolitan Ward 8 0.010066 0.936067 66
                          Farsidol Relocated 0.028815 0.840915 51
                 Sitapaila (SC-30) - GD Labs 0.152064 0.263227 56
